# **Experiment Notebook**

## Climate Comfort Index — Experiment 1: Regularised Linear Baseline

In [ ]:
# Do not modify this code
!pip install -q utstd

from utstd.ipyrenders import *

In [ ]:
# Do not modify this code
import warnings
warnings.simplefilter(action='ignore')

In [ ]:
import pandas as pd

pd.__version__

---
## Student Information

In [ ]:
student_name = "Abhishek Chopda"
student_id = "25611094"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

In [ ]:
# Standard data science stack.
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# Project toolkit, published to TestPyPI as `mlkit-aby`. All reusable
# functionality lives here so that this notebook contains only the
# decisions specific to this experiment.
from mlkit_aby.helpers import (
    eda,
    features,
    importance,
    modeling,
    targets,
    timeseries,
    visualization,
    weather,
)

# Project configuration: paths, modelling windows and forecast horizons.
from weather_intelligence.config import (
    CCI_HORIZONS,
    CCI_MODELS_DIR,
    CCI_TRAIN_START,
    PROCESSED_DATA_DIR,
    RAW_WEATHER_FILE,
)

from sklearn.linear_model import Ridge

pd.set_option("display.max_columns", 120)
sns.set_theme(style="whitegrid")

# Fixed throughout so that every result in this notebook is reproducible.
RANDOM_STATE = 42

print("mlkit-aby modules loaded")
print("Forecast horizons:", CCI_HORIZONS)
print("Modelling window starts:", CCI_TRAIN_START)

---
## A. Project Description

In [ ]:
business_use_case_description = """
Open Meteo is adding predictive intelligence to its weather platform. This
experiment builds the first of two services: a forecast of the Climate
Comfort Index (CCI) for Sydney, one, two and three days ahead.

The CCI condenses temperature, humidity, wind, cloud cover and rainfall into
a single 0-100 score describing how suitable a day is for being outdoors.
Raw weather variables already exist in the platform; what does not exist is
a single forward-looking number that a non-expert can act on. A tour
operator does not want five forecast variables, they want to know whether
Thursday is worth running the harbour cruise.

The service must predict the CCI of a future day using only information
available today, and expose that prediction through a REST API.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [ ]:
business_objectives = """
The objective is a comfort forecast accurate enough to support real
scheduling decisions three days ahead.

Accurate predictions let operators commit earlier: staff are rostered,
stock is ordered and marketing is scheduled against a reliable signal
rather than a guess, and customers receive advice they come to trust.

Inaccurate predictions carry asymmetric costs. Predicting comfort that does
not arrive is the more damaging error: customers are brought outdoors into
unpleasant conditions, refunds follow, and trust in the service is lost.
Predicting discomfort that does not arrive is cheaper, forgoing revenue on
a day that turned out fine. This asymmetry is not captured by a symmetric
error metric such as RMSE, and is noted as a limitation of this experiment.

Errors also matter more in some parts of the range than others. Confusing a
CCI of 82 with 78 moves a day from "excellent" to "comfortable" and changes
very little. Confusing 62 with 58 moves it across the boundary between
"comfortable" and "moderate", which may change a decision.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [ ]:
stakeholders_expectations_explanations = """
Predictions are consumed through a REST endpoint that accepts a date and
returns the CCI for the following three days.

Direct users are the developers integrating the endpoint into scheduling
tools, dashboards and consumer applications. They need stable response
shapes, honest error handling and transparency about model performance,
which the /model-metadata endpoint provides.

Indirectly affected are the operators making decisions on the forecast —
outdoor venues, event organisers, tourism operators, councils scheduling
maintenance — and the members of the public who act on advice derived from
it.

Expectations are shaped by a well-understood reality: everyone knows
weather forecasts are imperfect, and tolerance for error grows with the
horizon. A one-day forecast is expected to be dependable; a three-day
forecast is expected to be indicative. The service should be honest about
that gradient rather than presenting all three horizons as equally certain.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Raw Data

In [ ]:
# Hourly observations for Sydney (latitude -33.8688, longitude 151.2093),
# retrieved from the Open-Meteo Historical Weather API.
#
# If this file does not exist, regenerate it by running:
#     uv run python -m weather_intelligence.dataset
hourly = pd.read_csv(RAW_WEATHER_FILE, parse_dates=["time"])

print(f"{len(hourly):,} hourly observations")
print(f"{hourly['time'].min()} to {hourly['time'].max()}")
hourly.head()

In [ ]:
# Confirm the raw feed is complete before anything is derived from it.
eda.dataset_overview(hourly)

In [ ]:
eda.missing_summary(hourly)

### C.2 Define Target variable

In [ ]:
# The CCI formula is defined on daily weather values, so the hourly feed is
# aggregated first. The aggregation applied to each variable is a deliberate
# choice rather than a blanket average:
weather.TARGET_DAILY_AGGREGATIONS

In [ ]:
daily = weather.aggregate_to_daily(hourly)

print(f"{len(daily):,} daily observations")
daily.head()

In [ ]:
target_definition_explanations = """
The target is the Climate Comfort Index, a weighted combination of five
component scores covering temperature, humidity, wind, cloud and rain, each
scaled to 0-1 and combined into a 0-100 index. It is a regression target,
predicted at horizons of one, two and three days.

Two decisions shape how it is built.

First, the order of operations. Open-Meteo serves hourly observations, but
the index is defined per day, so aggregation has to happen somewhere. The
component formulas are non-linear — they use MAX, MIN and ABS — so
aggregating before and aggregating after give different answers. Aggregation
is applied first. The constants in the formulas settle this: precipitation
is divided by 20, which is a plausible daily rainfall total in millimetres
but almost never a single hour's rate. Computing hourly would leave that
term at essentially 1.0 every hour, making a quarter of the index dead
weight. This reading was confirmed by the subject coordinator.

Second, the aggregator per variable. Temperature, humidity, wind and cloud
use the daily mean, because comfort reflects typical conditions. Rainfall
uses the daily total, because what makes a day wet is how much fell, not
the average rate at which it fell.

The cost of using means is that within-day variation is discarded. A day
swinging between 12 C and 32 C and a day sitting flat at 22 C share a mean
and are not the same day. That information is not lost — it is recovered as
features in section F, rather than by complicating a target definition that
needs to stay simple and auditable.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [ ]:
# Component scores are computed separately from the index itself, so that
# the contribution of each can be inspected rather than taken on trust.
cci_components = targets.climate_comfort_components(daily)
cci_components.describe().T

In [ ]:
daily["cci"] = targets.climate_comfort_index(daily)

target_name = "cci"

daily[["date", "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
       "cloud_cover", "precipitation", "cci"]].head()

### C.4 Explore Target variable

In [ ]:
daily["cci"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["cci"], bins=50, edgecolor="white")
axes[0].set_title("Distribution of Climate Comfort Index")
axes[0].set_xlabel("CCI")
axes[0].set_ylabel("Number of days")

# Interpretation band boundaries from the project specification.
for boundary in [20, 40, 60, 80]:
    axes[0].axvline(boundary, linestyle="--", color="black", linewidth=1)

daily.set_index("date")["cci"].resample("ME").mean().plot(ax=axes[1], linewidth=1)
axes[1].set_title("Monthly mean CCI over time")
axes[1].set_xlabel("Date")
axes[1].set_ylabel("CCI")

plt.tight_layout()
plt.show()

In [ ]:
# Seasonality is the dominant structure in the target, so it is worth
# quantifying rather than only eyeballing the time series.
by_month = daily.assign(month=daily["date"].dt.month).groupby("month")["cci"]

fig, ax = plt.subplots(figsize=(11, 5))
daily.assign(month=daily["date"].dt.month).boxplot(
    column="cci", by="month", ax=ax, grid=False
)
ax.set_title("CCI by calendar month")
ax.set_xlabel("Month")
ax.set_ylabel("CCI")
plt.suptitle("")
plt.show()

by_month.mean().round(1)

In [ ]:
# How much of the index's variation does each component drive? A component
# that never varies cannot help a model discriminate between days.
component_summary = pd.DataFrame({
    "mean": cci_components.mean(),
    "std": cci_components.std(),
    "min": cci_components.min(),
    "max": cci_components.max(),
    "weight": pd.Series(targets.CCI_WEIGHTS),
})
component_summary["weighted_std"] = component_summary["std"] * component_summary["weight"]
component_summary.sort_values("weighted_std", ascending=False).round(3)

In [ ]:
target_distribution_explanations = """
The CCI is approximately normal, centred near 71 with a standard deviation
of about 11, and spans roughly 29 to 98 across the full record. No day in
26 years scores below 20, so the "very poor comfort" band is empty and the
"poor comfort" band nearly so. Sydney's climate is mild, and the index
reflects that: most days fall in the "comfortable" and "moderate" bands,
with a long left tail of storm days rather than a symmetric spread.

There is a clear seasonal signal, with comfort peaking in the shoulder
seasons and falling in mid-winter as temperature moves away from the 22 C
reference point. It would be easy to overstate its usefulness, and worth
resisting. The monthly means differ, but the spread *within* any given month
is far larger than the difference *between* months. Knowing it is April
narrows the expected CCI only slightly; the day-to-day weather within April
accounts for most of the variation. Section J.4 quantifies this directly:
a baseline predicting the historical average for each day of the year
achieves an R squared of only about 0.05. Seasonality is real structure, but
it is nowhere near sufficient on its own.

The component breakdown shows where the variation actually comes from.
Temperature carries the largest weight at 0.35 and varies substantially,
making it the dominant driver. Cloud score contributes meaningfully because
Sydney's cloud cover swings across the full 0-100% range. The rain score
behaves differently from the others: it sits at or near 1.0 on most days and
collapses only on the minority that are wet, so rather than contributing
steady variation it acts as an occasional sharp penalty. That skew is worth
remembering when reading the residuals, because those wet days are exactly
where a linear model is likely to struggle.

There is also a slow downward drift. Mean CCI has fallen by roughly four
points since 2000, from about 74 in the 2000s to about 70 in the 2020s.
This is the reason the modelling window for this target begins in 2015
rather than using the full record: a model deployed against 2026 conditions
should be trained on a climate resembling them.

A limitation worth stating plainly is that the index is a construct, not a
measurement. Its weights and reference points were specified, not learned,
so the model is being asked to predict a definition of comfort rather than
comfort itself. If the definition is wrong, an accurate model faithfully
reproduces the error.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `temperature_2m`

In [ ]:
# Daily mean temperature is the largest single contributor to the index, so
# its behaviour largely determines the target's behaviour.
daily["temperature_2m"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["temperature_2m"], bins=50, edgecolor="white")
axes[0].axvline(22, linestyle="--", color="black", label="CCI reference (22 C)")
axes[0].set_title("Distribution of daily mean temperature")
axes[0].set_xlabel("Temperature (C)")
axes[0].legend()

axes[1].scatter(daily["temperature_2m"], daily["cci"], alpha=0.15, edgecolor="none")
axes[1].axvline(22, linestyle="--", color="black")
axes[1].set_title("Same-day temperature against CCI")
axes[1].set_xlabel("Temperature (C)")
axes[1].set_ylabel("CCI")

plt.tight_layout()
plt.show()

In [ ]:
feature_1_insights = """
Daily mean temperature ranges from about 6 C to 32 C, centred near 18 C,
which sits four degrees below the 22 C comfort reference. Sydney is
therefore slightly cool relative to the index's definition of ideal, and
most of the temperature penalty comes from below rather than above.

The relationship with CCI is the inverted V the formula implies, but it is
noticeably asymmetric in practice. The right arm is sparse because few days
average much above 27 C, while the left arm is densely populated. The
consequence for modelling is important: across most of the observed range
the relationship between temperature and CCI is close to monotonic and
nearly linear, which is why a linear model is a credible starting point at
all. The non-linearity only bites on the small number of days averaging well
above the reference point, and because those days are rare their influence
on the fit is limited.

The scatter also shows substantial vertical spread at any given
temperature — a 20 C day can score anywhere from roughly 45 to 90. That
spread is the contribution of the other four components, and it is the part
of the variance a model must earn from the remaining features.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `cloud_cover`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(daily["cloud_cover"], bins=50, edgecolor="white")
axes[0].set_title("Distribution of daily mean cloud cover")
axes[0].set_xlabel("Cloud cover (%)")

axes[1].scatter(daily["cloud_cover"], daily["cci"], alpha=0.15, edgecolor="none")
axes[1].set_title("Same-day cloud cover against CCI")
axes[1].set_xlabel("Cloud cover (%)")
axes[1].set_ylabel("CCI")

plt.tight_layout()
plt.show()

daily["cloud_cover"].describe()

In [ ]:
feature_2_insights = """
Cloud cover spans nearly the full 0-100% range and is broadly spread rather
than concentrated, which makes it genuinely informative: a feature that
varies widely can discriminate between days in a way a near-constant one
cannot.

Its relationship with CCI is negative and close to linear, which follows
directly from the component formula, where the cloud score is a straight
linear function of cover with no clipping in the observed range. Of the five
components this is the best behaved from a linear model's point of view.

Cloud cover is also correlated with the other components, most obviously
rainfall — wet days are cloudy days. That correlation is useful for
prediction but means the coefficients of cloud and rain terms should not be
read as independent causal effects, since the two carry overlapping
information.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.7 Explore Feature of Interest `precipitation`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Rainfall is heavily right skewed, so a log scale on the count axis is the
# only way to see the tail at all.
axes[0].hist(daily["precipitation"], bins=60, edgecolor="white")
axes[0].set_yscale("log")
axes[0].axvline(20, linestyle="--", color="black", label="CCI rain floor (20 mm)")
axes[0].set_title("Distribution of daily rainfall (log count)")
axes[0].set_xlabel("Precipitation (mm)")
axes[0].legend()

axes[1].scatter(daily["precipitation"], daily["cci"], alpha=0.15, edgecolor="none")
axes[1].axvline(20, linestyle="--", color="black")
axes[1].set_title("Same-day rainfall against CCI")
axes[1].set_xlabel("Precipitation (mm)")
axes[1].set_ylabel("CCI")

plt.tight_layout()
plt.show()

print("Days with no rain: {:.1%}".format((daily["precipitation"] == 0).mean()))
print("Days above the 20 mm floor: {:.1%}".format((daily["precipitation"] >= 20).mean()))
daily["precipitation"].describe()

In [ ]:
feature_n_insights = """
Rainfall is the most awkwardly distributed variable in the dataset. Slightly
over half of all days record no rain at all, and the distribution decays
steeply from there, with a thin tail reaching past 130 mm. The mean is
around 3 mm while the median is zero — a distribution a linear model handles
poorly, because the majority of its mass sits at a single point.

Its effect on CCI is a clipped ramp. Rainfall above 20 mm drives the rain
score to zero, and beyond that point additional rain changes the index not
at all. Around 2% of days sit in that saturated region. A linear model
cannot represent a clipped relationship and will keep extrapolating the
penalty downwards, under-predicting comfort on the wettest days.

This is also where the aggregation decision made in section C.2 pays off.
Computed hourly, rainfall would almost never approach 20 mm within a single
hour, the rain score would sit at 1.0 nearly always, and 15% of the index
weight would carry no information. As a daily total it varies meaningfully
and genuinely separates wet days from dry ones.

For prediction, the more useful signal is likely to be whether it has been
raining recently rather than the exact millimetre total, which motivates the
binary-like and rolling rainfall features constructed in section F.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest

> Further variables (humidity, wind speed) follow the same pattern and are examined through the correlation analysis in section D rather than individually here.

In [ ]:
# A compact view of the remaining candidate variables against the target,
# rather than repeating the full treatment for each.
candidate_variables = [
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
    "cloud_cover", "precipitation", "wind_gusts_10m",
]
eda.numerical_summary(daily[candidate_variables]).round(2)

---
## D. Feature Selection

### D.1 Approach "Correlation with the future target"

In [ ]:
# The question is not which variables correlate with *today's* CCI — by
# construction they all do, since today's CCI is computed from them. The
# question is which of today's variables carry information about a *future*
# day's CCI. That is what a forecast actually needs.
horizon_correlations = {}

for h in CCI_HORIZONS:
    future_cci = daily["cci"].shift(-h)
    horizon_correlations[f"cci_h{h}"] = (
        daily[candidate_variables + ["cci"]].corrwith(future_cci)
    )

correlation_with_future = pd.DataFrame(horizon_correlations)
correlation_with_future["abs_h1"] = correlation_with_future["cci_h1"].abs()
correlation_with_future.sort_values("abs_h1", ascending=False).drop(columns="abs_h1").round(3)

In [ ]:
feature_selection_1_insights = """
This approach ranks candidate variables by their correlation with the CCI
one, two and three days ahead, which is the quantity the model must
actually predict.

Two results stand out. First, today's CCI is itself the strongest single
predictor of tomorrow's, at roughly 0.5. Weather persists, and this is a
legitimate feature rather than leakage: today's CCI is computed entirely
from today's weather, which is known at prediction time. Second, and more
importantly, every correlation decays as the horizon lengthens. The link
between today's conditions and the third day ahead is materially weaker
than the link to tomorrow. This is the forecast skill decay that any
honest weather model shows, and it predicts in advance that the three-day
model will perform worse than the one-day model.

Among the raw weather variables, temperature and cloud cover carry the most
signal, consistent with their weights and variability in the index.
Rainfall correlates weakly with future comfort despite mattering a great
deal to same-day comfort, which makes sense: rain is far less persistent
day to day than temperature is.

The limitation of this approach is that Pearson correlation measures linear
association only. A variable with a real but non-linear relationship —
temperature is the obvious candidate, given its inverted V shape — will be
understated here. It is used as a screening tool, not as the sole basis for
exclusion.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### D.2 Approach "Multicollinearity screening"

In [ ]:
# Highly correlated inputs destabilise linear models: the fit can shift large
# offsetting coefficients between near-duplicate features, which makes the
# coefficients unreliable to interpret even when predictions are fine.
eda.find_highly_correlated_features(daily[candidate_variables], threshold=0.70)

In [ ]:
correlation_matrix = eda.calculate_correlation_matrix(daily[candidate_variables + ["cci"]])
visualization.plot_correlation_heatmap(
    correlation_matrix,
    figsize=(9, 7),
    title="Correlation between candidate daily weather variables",
)

In [ ]:
feature_selection_2_insights = """
This approach screens the candidate variables for redundancy with each
other, which matters because the model chosen for this experiment is
linear and linear models are sensitive to collinear inputs.

The clearest redundancy is between wind speed and wind gusts, which measure
the same physical phenomenon at different statistics and correlate strongly.
Keeping both would split one effect across two coefficients without adding
information. Wind speed is retained because it is the variable the CCI
formula actually uses; gusts are retained as a candidate for the hazard
model rather than this one.

Cloud cover, humidity and rainfall form a looser cluster, all reflecting how
wet and overcast conditions are. Their correlations are moderate rather than
severe, and each contributes a distinct component to the index, so all three
are kept. Ridge regression is used in section J partly for this reason: the
L2 penalty stabilises the fit when inputs share information, which is a
more graceful response to moderate collinearity than dropping variables that
carry real signal.

No variable is excluded on the basis of this screen alone. It informs the
choice of algorithm more than it informs the choice of features.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

## D.z Final Selection of Features

In [ ]:
# The five variables the CCI formula is built from, which are therefore the
# variables carrying direct information about it. Wind gusts are dropped as
# redundant with wind speed for this target.
features_list = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
]

# Today's CCI is a legitimate and strong predictor of a future CCI, since it
# is computed only from weather already observed.
base_features = features_list + ["cci"]

base_features

In [ ]:
feature_selection_explanations = """
The selected base variables are the five the CCI formula is constructed
from, plus the CCI itself as a persistence signal.

The reasoning is that these five variables are not merely correlated with
the target, they define it. A perfect three-day forecast of these five
variables would yield a perfect CCI forecast by construction. The modelling
problem is therefore really a weather forecasting problem wearing an index
as a wrapper, and the most direct path to predicting the index is to track
the variables that compose it.

Wind gusts are excluded as redundant with wind speed, which the formula
actually uses. Snowfall is excluded because it is identically zero across
the entire 26-year record for Sydney and can carry no information.

These are base variables rather than final model inputs. Section F derives
lagged, rolling and calendar features from them, and the model is trained on
those derivatives. Selection here is about which underlying signals to
track; the feature engineering step decides how far back to look.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation: Missing value check

In [ ]:
eda.missing_summary(daily)

In [ ]:
data_cleaning_1_explanations = """
The daily dataset contains no missing values in any column.

This is expected rather than lucky. Open-Meteo's historical archive is a
reanalysis product: observations are assimilated into a physical model that
produces a complete, gap-free grid. There are no station outages or
unreported days to handle.

The check is still performed and documented, for two reasons. Any missing
hourly value would propagate silently into a daily aggregate — pandas
aggregation skips missing values by default, so a day with only six reported
hours would produce a mean computed from six hours while looking identical
to a complete one. Confirming completeness at the source rules that out. It
also establishes that missing values appearing later in the pipeline must
have been introduced by the feature engineering step, where they are
expected and handled deliberately, rather than inherited from the data.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation: Calendar continuity check

In [ ]:
# Lag and rolling features assume consecutive rows are consecutive days. If
# a date were missing, a "1 day lag" would silently reach further back.
expected_dates = pd.date_range(daily["date"].min(), daily["date"].max(), freq="D")
missing_dates = expected_dates.difference(daily["date"])

print(f"Expected days: {len(expected_dates):,}")
print(f"Observed days: {len(daily):,}")
print(f"Missing days:  {len(missing_dates)}")
print(f"Duplicate dates: {daily['date'].duplicated().sum()}")

In [ ]:
data_cleaning_2_explanations = """
The date series is complete and strictly consecutive, with no gaps and no
duplicates across the full record.

This check matters more than it might appear. Every lag and rolling feature
built in section F is positional: they shift rows, not dates. If a single
day were missing from the middle of the series, a feature labelled "one day
ago" would silently be reaching two days back at that point, and a rolling
seven-day window would be spanning eight calendar days. The error would not
raise an exception or produce missing values — it would quietly corrupt a
handful of rows in a way that is very difficult to detect downstream.

Confirming continuity here is what licenses the use of positional shifts
later. Had gaps been present, the correct response would have been to
reindex the frame onto a complete date range first, inserting explicit
missing values that the feature functions would then propagate visibly.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation: Physical plausibility check

In [ ]:
# Reanalysis output can still contain physically impossible values. Checking
# the bounds is cheap and catches unit errors and corrupt records.
plausibility = pd.DataFrame({
    "min": daily[candidate_variables].min(),
    "max": daily[candidate_variables].max(),
})
plausibility["plausible_range"] = [
    "-10 to 50 C", "0 to 100 %", "0 to 150 km/h",
    "0 to 100 %", "0 to 500 mm", "0 to 250 km/h",
]

print("CCI within its defined 0-100 range:",
      bool(daily["cci"].between(0, 100).all()))
plausibility

In [ ]:
data_cleaning_3_explanations = """
Every variable falls inside a physically plausible range, and the computed
CCI lies within its defined 0-100 bounds for every one of the 9,497 days.

Percentage variables stay within 0-100, temperature spans roughly 6 C to
32 C as daily means, and rainfall peaks at 138 mm on the February 2020 east
coast low — extreme, but a real recorded event rather than a data error.

The CCI bounds check is the more important of the two. The index is
constructed by code, not measured, so a bug in the component formulas — a
clipping step omitted, a divisor wrong — would most likely surface as values
outside 0-100. Confirming the range holds across every observation is a
direct end-to-end test that the target has been built as specified. The same
property is asserted in the package's unit tests against hand-calculated
values; this repeats the check against real data at full scale.

No transformation is applied as a result of this check. It is a validation
step, and the appropriate outcome is that nothing needed fixing.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

---
## F. Feature Engineering

### F.1 New Feature "Lagged weather observations"

In [ ]:
# The experiment-specific decision is *which* features to build. That
# decision is written down as a recipe, and the package assembles the
# features from it.
#
# Keeping the recipe as data rather than as code matters for deployment: it
# is saved inside the model artefact, so the API builds its features from
# the same recipe the model was trained on and the two cannot drift apart.
CCI_RECIPE = {
    "base_variables": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "cloud_cover",
        "precipitation",
        "cci",
    ],
    "lags": [1, 2, 3, 7],
    "windows": [3, 7, 30],
    "statistics": ["mean", "std"],
    "derived_indices": ["cci"],
    "within_day": True,
    "within_day_lags": [1],
}

daily_features = features.build_daily_features(hourly, CCI_RECIPE)

print(f"{daily_features.shape[1] - 1} features built for {len(daily_features):,} days")
print(f"History required per prediction: {features.required_history_days(CCI_RECIPE)} days")

# Lag features: earlier days' values brought onto the current row. Every lag
# is strictly positive, so no feature can see beyond its own date.
lag_columns = [c for c in daily_features.columns if "_lag" in c]
print(f"{len(lag_columns)} lag features")

daily_features[["date", "cci", "cci_lag1", "cci_lag2", "temperature_2m_lag1"]].head(8)

In [ ]:
feature_engineering_1_explanations = """
Lagged copies of each base variable at one, two, three and seven days give
the model access to the recent trajectory of the weather rather than a
single snapshot.

The motivation is that direction matters as much as level. A 19 C day
following three warmer days sits in a cooling pattern; the same 19 C day
following three cooler days sits in a warming one. Those two days look
identical to a model given only today's value, and they imply different
things about tomorrow. Weather systems move through Sydney over a few days,
so the recent sequence carries genuine information about what arrives next.

The specific lags are chosen to cover that structure at low cost: one, two
and three days capture the passage of a typical frontal system, and seven
days provides a same-day-last-week reference that is far enough back to
reflect the broader synoptic pattern rather than immediate persistence.

These features are safe by construction. A positive shift can only bring
past values forward, and the package rejects a lag of zero or less outright,
so the common mistake of writing a negative shift cannot silently create a
model that reads the future.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "Rolling window statistics"

In [ ]:
# Rolling window features: summaries of the recent past at three
# timescales. Windows close at the current row, so every observation inside
# them is already known.
rolling_columns = [c for c in daily_features.columns if "_roll" in c]
print(f"{len(rolling_columns)} rolling features")

daily_features[["date", "cci", "cci_roll3_mean", "cci_roll7_mean",
                "cci_roll7_std", "cci_roll30_mean"]].tail(5)

In [ ]:
feature_engineering_2_explanations = """
Rolling means and standard deviations over three, seven and thirty days
summarise the recent past at three different timescales.

The means serve distinct purposes. A three-day mean smooths out a single
unrepresentative day. A seven-day mean describes the current weather
pattern. A thirty-day mean acts as a local seasonal level — it encodes
roughly where in the year the series currently sits, learned from the data
rather than imposed by the calendar, and adapts automatically to an unusually
warm or cool season.

The standard deviations add something the means cannot: how settled
conditions are. A week of stable weather and a week of violent swings can
share a mean while implying very different confidence about tomorrow. High
recent variability is itself a signal that the near future is less
predictable.

Incomplete windows are deliberately left as missing rather than computed
from partial history. A "thirty-day mean" calculated from four days is not a
thirty-day mean, and training on such values would teach the model a
relationship that does not hold anywhere else in the series. Those rows are
dropped before modelling, costing the first twenty-nine days of the record.
Because features are built on the full history from 2000 and the modelling
window only begins in 2015, this warm-up is absorbed entirely by data that
would not have been used regardless.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "Cyclical calendar encoding"

In [ ]:
# Demonstrating why the calendar encoding is cyclical: the last day of the
# year and the first day of the next must sit next to each other, not 364
# apart.
year_boundary = daily_features[
    daily_features["date"].isin(
        pd.to_datetime(["2024-12-30", "2024-12-31", "2025-01-01", "2025-01-02"])
    )
]
year_boundary[["date", "day_of_year_sin", "day_of_year_cos"]].round(4)

In [ ]:
feature_engineering_n_explanations = """
Day of year and month are encoded as paired sine and cosine values rather
than as raw integers.

Seasonality is the dominant structure in the CCI, so the model needs to know
where in the year a day falls. The difficulty is that a raw day-of-year
number is a poor way to express that. It tells a model that day 365 and day
1 are 364 units apart, when in seasonal terms they are consecutive. Any
model treating that number as a continuous quantity will learn a
discontinuity at the new year that does not exist in the weather.

The sine and cosine pair places each date on a circle, so the distance
between two dates reflects how far apart they actually are in the seasonal
cycle. The printed output above shows this directly: the values move
smoothly across the 31 December to 1 January boundary rather than jumping.
Two components are required because either one alone is ambiguous — a single
sine value cannot distinguish spring from autumn, since both sit at the same
height on the curve.

The divisor is 366 rather than 365 so that leap years do not push the
encoding past a full revolution and misalign against other years.

These features are the only ones in the model that are known with certainty
for any future date, which makes them the backbone of the longer horizons
where recent weather has less to say.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.4 New Feature "Within-day variation"

In [ ]:
# The target uses daily means, which discard the spread within a day. That
# information is recovered here as features, where it can help without
# complicating the target definition.
daily_features[["date", "temperature_2m", "temperature_2m_min",
                "temperature_2m_max", "temperature_range"]].head()

In [ ]:
# These are lagged by one day as well, so the model sees yesterday's spread
# in addition to today's.
print("Total columns after feature engineering:", daily_features.shape[1])

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
# Feature engineering was performed on the full record from 2000 so that
# rolling windows are complete from the first modelled day. Only now is the
# data restricted to the modelling window chosen for this target.
model_frame = daily_features[
    daily_features["date"] >= CCI_TRAIN_START
].reset_index(drop=True)

print(f"Modelling window: {model_frame['date'].min().date()} to {model_frame['date'].max().date()}")
print(f"{len(model_frame):,} days")

In [ ]:
# Columns that must never be used as inputs: the date itself, and the target.
EXCLUDED_COLUMNS = {"date", "target_date", "cci_h1", "cci_h2", "cci_h3"}

feature_columns = [c for c in model_frame.columns if c not in EXCLUDED_COLUMNS]

print(f"{len(feature_columns)} candidate input features")
print("Today's CCI retained as an input:", "cci" in feature_columns)

In [ ]:
# Split boundaries. Roughly eight years to train on, then eighteen months
# each for validation and test.
VALIDATION_START = "2023-01-01"
TEST_START = "2024-07-01"

# One supervised dataset per horizon: identical features, different target.
datasets = {}

for h in CCI_HORIZONS:
    supervised = timeseries.make_supervised_frame(model_frame, "cci", horizon=h)

    # Rows whose rolling windows were incomplete are dropped here. Because
    # features were built from 2000, this removes nothing inside the window.
    supervised = supervised.dropna(subset=feature_columns).reset_index(drop=True)

    train, validation, test = timeseries.chronological_split(
        supervised,
        validation_start=VALIDATION_START,
        test_start=TEST_START,
        horizon=h,
    )
    datasets[h] = {"train": train, "validation": validation, "test": test}

    print(f"horizon {h}:  train {len(train):>5}   validation {len(validation):>4}   test {len(test):>4}")

In [ ]:
# Confirm the embargo worked: no training target may fall on or after the
# validation period, and no validation target on or after the test period.
for h in CCI_HORIZONS:
    d = datasets[h]
    assert d["train"]["target_date"].max() < pd.Timestamp(VALIDATION_START)
    assert d["validation"]["target_date"].max() < pd.Timestamp(TEST_START)
    assert d["train"]["date"].max() < d["validation"]["date"].min()
    assert d["validation"]["date"].max() < d["test"]["date"].min()

print("Embargo verified: no target crosses a split boundary.")

In [ ]:
data_splitting_explanations = """
The data is split chronologically: the earliest period trains, the middle
validates and the most recent period tests.

A random split would be actively wrong here, and the reason is worth being
precise about. Weather is strongly autocorrelated — tomorrow closely
resembles today. A random split places 3 March in training and 4 March in
test, which are near-duplicates of one another. The model is effectively
shown the answers, test scores look excellent, and the moment it is deployed
against genuinely unseen future days the performance collapses. A
chronological split mirrors how the service is actually used: learn from the
past, predict forward.

An embargo is applied at each boundary, removing the final `horizon` days
from the training and validation sets. Without it the last training row,
dated one day before the validation period, would carry a target inside that
period, leaking across the boundary. The cost is a handful of rows and the
assertion above confirms it worked.

The boundaries give roughly eight years of training data with eighteen
months each for validation and test. The test period is the most recent
eighteen months precisely because that is the hardest and most honest test:
it is the data closest to the conditions the deployed service will face.

One consequence should be stated. Each split covers whole seasonal cycles,
which matters when seasonality dominates the target. A test period of only a
few months could land entirely in summer and produce a score that says more
about which months were selected than about the model.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "Feature standardisation"

In [ ]:
# Scaling is not applied as a separate step here. It is made the first stage
# of a scikit-learn Pipeline, so that the scaler and the model are fitted,
# saved and applied as one object.
#
# The datasets below therefore hold *unscaled* features. The pipeline scales
# them internally using statistics learned from the training split alone.
def make_model(alpha):
    """Build an unfitted scaler + ridge pipeline for a given alpha."""
    return modeling.make_scaled_pipeline(
        Ridge(alpha=alpha, random_state=RANDOM_STATE)
    )


for h in CCI_HORIZONS:
    d = datasets[h]
    target = f"cci_h{h}"

    d["X_train"] = d["train"][feature_columns]
    d["X_val"] = d["validation"][feature_columns]
    d["X_test"] = d["test"][feature_columns]

    d["y_train"] = d["train"][target]
    d["y_val"] = d["validation"][target]
    d["y_test"] = d["test"][target]

# Confirming the pipeline standardises as expected, using the one-day model.
demonstration = make_model(alpha=1.0).fit(datasets[1]["X_train"], datasets[1]["y_train"])
scaled_preview = demonstration.named_steps["scaler"].transform(datasets[1]["X_train"])
pd.DataFrame(scaled_preview, columns=feature_columns)[
    ["temperature_2m", "precipitation"]
].describe().loc[["mean", "std"]].round(3)

In [ ]:
data_transformation_1_explanations = """
All input features are standardised to zero mean and unit variance, applied
as the first stage of a scikit-learn Pipeline rather than as a separate
step.

Two reasons for standardising. Ridge regression applies an L2 penalty to the
size of the coefficients, and the size of a coefficient depends on the units
of its feature. Rainfall in millimetres and humidity in percent occupy very
different numeric ranges, so an unscaled penalty would shrink them by
arbitrarily different amounts — the regularisation would be determined by
measurement units rather than by the data. Standardising puts every feature
on a common footing so the penalty applies evenly. Separately, standardised
coefficients become directly comparable to one another, which is what makes
the coefficient-based feature importance in section J.4 interpretable at all.

The scaler is fitted on the training set alone and then applied unchanged to
validation and test. Fitting on all the data before splitting is a subtle
and common form of leakage: the means and standard deviations would encode
information about the test period, and in a time series that means encoding
information from the future. The effect is usually small but it is real, and
it inflates the reported score in exactly the direction that flatters the
model. Using a Pipeline enforces this automatically, because fitting the
pipeline fits the scaler on whatever data the pipeline is fitted to.

Bundling the scaler with the model also matters at deployment. Predictions
made on unscaled inputs, or on inputs scaled with different statistics, are
not slightly worse — they are wrong. Saving the two as one object means the
artefact cannot be used incorrectly, because there is no separate scaling
step for the API to forget or to apply differently.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

---
## H. Save Datasets

In [ ]:
data_final_path = "../../data/processed"
Path(data_final_path).mkdir(parents=True, exist_ok=True)

# The template's save cell below expects a single set of names. The one-day
# horizon is used for those; all three horizons are saved explicitly here.
for h in CCI_HORIZONS:
    d = datasets[h]
    d["X_train"].to_csv(f"{data_final_path}/cci_h{h}_X_train.csv", index=False)
    d["y_train"].to_csv(f"{data_final_path}/cci_h{h}_y_train.csv", index=False)
    d["X_val"].to_csv(f"{data_final_path}/cci_h{h}_X_val.csv", index=False)
    d["y_val"].to_csv(f"{data_final_path}/cci_h{h}_y_val.csv", index=False)
    d["X_test"].to_csv(f"{data_final_path}/cci_h{h}_X_test.csv", index=False)
    d["y_test"].to_csv(f"{data_final_path}/cci_h{h}_y_test.csv", index=False)

print(f"Saved datasets for horizons {CCI_HORIZONS}")

# Bind the template's expected names to the one-day horizon.
X_train, y_train = datasets[1]["X_train"], datasets[1]["y_train"]
X_val, y_val = datasets[1]["X_val"], datasets[1]["y_val"]
X_test, y_test = datasets[1]["X_test"], datasets[1]["y_test"]

In [ ]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(f'{data_final_path}/X_train.csv', index=False)
  y_train.to_csv(f'{data_final_path}/y_train.csv', index=False)

  X_val.to_csv(f'{data_final_path}/X_val.csv', index=False)
  y_val.to_csv(f'{data_final_path}/y_val.csv', index=False)

  X_test.to_csv(f'{data_final_path}/X_test.csv', index=False)
  y_test.to_csv(f'{data_final_path}/y_test.csv', index=False)
except Exception as e:
  print(e)

---
## I. Selection of Performance Metrics

In [ ]:
# Root mean squared error, mean absolute error and the coefficient of
# determination, all provided by the project package.
help(modeling.evaluate_regression)

In [ ]:
performance_metrics_explanations = """
Three metrics are reported: RMSE, MAE and R squared.

MAE is the headline. It is the average absolute error in CCI points, which
makes it directly interpretable against the index's own interpretation
bands: an MAE of 5 means a typical forecast lands within five points of the
truth, which is less than the width of a single comfort band. That is a
statement a non-technical stakeholder can act on without translation.

RMSE is reported alongside it because it squares the errors before
averaging, and therefore penalises large misses far more heavily than small
ones. That matters for this use case. A forecast that is usually excellent
but occasionally predicts a comfortable day that turns out to be a storm
does more damage than its average error suggests, and comparing RMSE against
MAE exposes that behaviour: a gap between the two indicates the error
distribution has a heavy tail rather than being uniformly spread.

R squared measures the proportion of variance explained and is used to
compare against baselines rather than as a target in its own right. It is
reported with a caveat: because the CCI is strongly seasonal, a model that
learns nothing but the time of year will already post a respectable R
squared. Judged against zero the number flatters; judged against a seasonal
baseline it is informative. Section J.4 therefore reports it against both.

Accuracy-style metrics and classification of the CCI into its interpretation
bands are deliberately not used. The target is continuous, and banding it
would discard the distinction between a near miss and a large error at a
boundary.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='performance_metrics_explanations', value=performance_metrics_explanations)

## J. Train Machine Learning Model

### J.1 Import Algorithm

In [ ]:
# Ridge regression: ordinary least squares with an L2 penalty on the
# coefficients.
help(modeling.train_ridge_regression)

In [ ]:
algorithm_selection_explanations = """
This experiment uses ridge regression, chosen deliberately as a baseline
rather than as the expected final model.

The case for starting linear is that the target is close to linear by
construction. The CCI is a weighted sum of five component scores, and four
of those five are linear functions of their weather variable across most of
the observed range. If the model could see the five variables on the target
day, a linear model would reproduce the index almost exactly. It cannot see
them — that is the forecasting problem — but the structure of the target
still makes a linear relationship a reasonable first hypothesis rather than
a naive one.

Ridge is preferred over ordinary least squares because of the feature set.
Lagged and rolling versions of the same variable are strongly correlated
with one another by construction: a three-day rolling mean of temperature
and a one-day lag of temperature share most of their information. Ordinary
least squares responds to this badly, distributing large offsetting
coefficients across near-duplicate features, which makes the fit unstable
and the coefficients meaningless. The L2 penalty shrinks correlated
coefficients towards each other instead, which is a far more graceful
response than dropping features that carry genuine signal.

Its known limitation is the reason this is experiment one and not the final
model. A linear model cannot represent the clipping in the rain component,
nor the inverted V of the temperature component, so it is expected to
under-perform specifically on hot days and very wet days. Establishing by
how much is the point of running it: the next experiment uses a tree
ensemble, and without this baseline there would be no way to say whether the
extra complexity bought anything.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

In [ ]:
# Alpha controls the strength of the L2 penalty: larger values shrink
# coefficients harder, trading variance for bias. It is selected on the
# validation set, separately for each horizon.
ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

alpha_search = []

for h in CCI_HORIZONS:
    d = datasets[h]
    for alpha in ALPHA_GRID:
        candidate = make_model(alpha).fit(d["X_train"], d["y_train"])
        metrics = modeling.evaluate_regression(
            d["y_val"], candidate.predict(d["X_val"])
        )
        alpha_search.append({"horizon": h, "alpha": alpha, **metrics})

alpha_search = pd.DataFrame(alpha_search)
alpha_search.pivot(index="alpha", columns="horizon", values="mae").round(3)

In [ ]:
# The best alpha per horizon, by validation MAE.
best_alphas = (
    alpha_search.loc[alpha_search.groupby("horizon")["mae"].idxmin()]
    .set_index("horizon")["alpha"]
    .to_dict()
)
best_alphas

In [ ]:
hyperparameters_selection_explanations = """
Ridge has one hyperparameter of consequence, alpha, which sets the strength
of the L2 penalty. It is tuned over a logarithmic grid from 0.01 to 1000,
because the useful range of a regularisation parameter spans orders of
magnitude and a linear grid would waste nearly all its points in one region.

Selection is made on the validation set rather than by cross-validation.
Standard k-fold cross-validation shuffles observations across folds, which
breaks the temporal ordering and reintroduces exactly the leakage the
chronological split was designed to prevent. A dedicated validation period
that sits between training and test preserves the ordering and keeps the
test set genuinely untouched until the final evaluation.

Alpha is tuned separately for each horizon. This is one of the practical
advantages of training three independent models: the amount of
regularisation that suits a one-day forecast need not suit a three-day one.
The expectation is that longer horizons prefer stronger shrinkage, because
the recent-weather features carry less signal there and shrinking them
harder pushes the model towards the seasonal features that remain reliable.

No other hyperparameters are tuned. Ridge has no others that materially
affect the fit, which is part of its appeal as a baseline: there is very
little room to accidentally tune the model to the validation set.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
# One pipeline per horizon, each using its own selected alpha.
models = {}

for h in CCI_HORIZONS:
    d = datasets[h]
    models[h] = make_model(best_alphas[h]).fit(d["X_train"], d["y_train"])
    print(f"horizon {h}: fitted scaler + ridge (alpha={best_alphas[h]}) "
          f"on {len(d['X_train']):,} days x {len(feature_columns)} features")

### J.4 Model Technical Performance

In [ ]:
# Two reference points, because a model must beat something to be useful.
#
# Persistence: predict that the future CCI equals today's CCI. This is the
# naive forecast a person makes by looking out of the window.
#
# Seasonal mean: predict the average CCI historically observed on that day
# of the year. This is what a model learns if it learns only seasonality.
seasonal_mean = (
    datasets[1]["train"]
    .assign(day_of_year=lambda f: f["date"].dt.dayofyear)
    .groupby("day_of_year")["cci"].mean()
)

results = []

for h in CCI_HORIZONS:
    d = datasets[h]
    test_rows = d["test"]

    predictions = {
        "Ridge regression": models[h].predict(d["X_test"]),
        "Persistence baseline": test_rows["cci"].to_numpy(),
        "Seasonal mean baseline": test_rows["date"].dt.dayofyear.map(seasonal_mean).to_numpy(),
    }

    for name, y_pred in predictions.items():
        results.append({
            "horizon": h,
            "model": name,
            **modeling.evaluate_regression(d["y_test"], y_pred),
        })

results = pd.DataFrame(results)
results.pivot(index="model", columns="horizon", values=["mae", "rmse", "r2"]).round(3)

In [ ]:
# Forecast skill decay: the same model gets harder as the horizon grows.
fig, ax = plt.subplots(figsize=(9, 5))

for name, group in results.groupby("model"):
    ax.plot(group["horizon"], group["mae"], marker="o", label=name)

ax.set_xticks(CCI_HORIZONS)
ax.set_xlabel("Forecast horizon (days ahead)")
ax.set_ylabel("Test MAE (CCI points)")
ax.set_title("Forecast error by horizon")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostics for the one-day model.
d = datasets[1]
y_pred_h1 = models[1].predict(d["X_test"])

visualization.plot_predicted_vs_actual(
    d["y_test"], y_pred_h1, title="CCI one day ahead: predicted vs actual (test set)"
)
plt.show()

visualization.plot_residuals(
    d["y_test"], y_pred_h1, title="CCI one day ahead: residuals (test set)"
)
plt.show()

In [ ]:
visualization.plot_forecast_timeline(
    d["test"]["target_date"], d["y_test"], y_pred_h1,
    title="CCI one day ahead: forecast against actual over the test period",
)
plt.show()

In [ ]:
# Which features does the model actually rely on? The fitted estimator is
# reached through the pipeline, and its coefficients are comparable because
# the pipeline standardised the inputs.
coefficients = importance.linear_coefficient_importance(
    models[1].named_steps["model"], feature_columns
)

visualization.plot_feature_importance(
    coefficients, value_column="abs_coefficient", top_n=15,
    title="CCI one day ahead: largest standardised coefficients",
)
plt.show()

coefficients.head(15).round(3)

In [ ]:
# Permutation importance measures what the model relies on when
# generalising, computed on held-out data rather than the training set.
permutation = importance.permutation_feature_importance(
    models[1], d["X_test"], d["y_test"], n_repeats=10, random_state=RANDOM_STATE
)
permutation.head(15).round(4)

In [ ]:
# Where does the model fail? Errors are examined against the conditions on
# the day being predicted, to test the expectations set in section C.
error_analysis = d["test"][["date", "target_date"]].copy()
error_analysis["actual"] = d["y_test"].to_numpy()
error_analysis["predicted"] = y_pred_h1
error_analysis["error"] = error_analysis["actual"] - error_analysis["predicted"]

# Conditions on the predicted day, joined back from the daily record.
conditions = daily.set_index("date")[["temperature_2m", "precipitation"]]
error_analysis = error_analysis.join(conditions, on="target_date")

wet = error_analysis["precipitation"] > 10
hot = error_analysis["temperature_2m"] > 25

print(f"MAE on all test days:      {error_analysis['error'].abs().mean():.2f}")
print(f"MAE on wet days (>10 mm):  {error_analysis.loc[wet, 'error'].abs().mean():.2f}  (n={wet.sum()})")
print(f"MAE on hot days (>25 C):   {error_analysis.loc[hot, 'error'].abs().mean():.2f}  (n={hot.sum()})")
print()
print("Largest over-predictions (model said comfortable, day was not):")
error_analysis.nsmallest(5, "error")[
    ["target_date", "actual", "predicted", "error", "precipitation"]
].round(1)

In [ ]:
model_performance_explanations = """
The ridge model beats both baselines at every horizon. Test MAE is 6.49 CCI
points at one day, 7.31 at two and 7.50 at three, against a persistence
baseline of 7.56, 9.10 and 9.59 and a seasonal baseline of about 8.70 at
every horizon.

The two baselines fail in opposite and instructive ways. Persistence is
competitive at one day and collapses beyond it, posting a negative R squared
at two and three days, meaning that predicting "the same as today" is
actually worse than predicting the long-run average once you look more than
a day ahead. The seasonal baseline is flat across horizons, as it must be
since it ignores current conditions entirely, but its R squared of roughly
0.05 shows that the time of year alone explains very little day-to-day
variation. The model's value comes from combining the two sources: recent
conditions, which decay in usefulness, and seasonal position, which does not.

That combination is visible in the coefficients. At one day ahead the
dominant input is today's CCI by a wide margin, with a standardised
coefficient of 2.99 against 1.29 for the next feature, and permutation
importance of 0.187 against 0.038. By three days ahead the ranking has
changed: the thirty-day rolling mean of CCI overtakes today's value, and the
cyclical calendar features climb from around rank 16 to around rank 7. The
model shifts from persistence to climatology as the horizon lengthens,
which is exactly the behaviour meteorological forecasting shows and was not
imposed by hand.

Forecast skill decays with horizon, and R squared falls from 0.45 to 0.29 to
0.25. This is a property of the atmosphere rather than a deficiency in the
model, and it is the single most important thing to communicate to users:
the three-day figure is indicative where the one-day figure is dependable.

RMSE sits well above MAE at every horizon (8.54 against 6.49 at one day),
indicating a heavy-tailed error distribution — most days are forecast well
and a minority very badly. The error analysis locates that tail precisely.
On dry days MAE is 5.67 and on days with light rain 5.63, but on days with
more than 10 mm it is 15.97, nearly three times worse. On those wet days the
model over-predicts comfort 98% of the time, with a mean error of -15.8
points.

This is the predicted failure of a linear model on a clipped component, and
it is worth being clear that it is not a general bias. Overall mean error is
-0.14, essentially zero, because a small positive bias on the 504 dry days
offsets a large negative one on the 44 wet days. The model is not
systematically optimistic; it is systematically optimistic about rain
specifically, which is the more commercially damaging direction of error
identified in section A.

One expectation was not borne out. Hot days were anticipated to be a second
failure mode, given the inverted V in the temperature component, but MAE on
days above 25 C is 4.58, better than the overall average. Such days are both
rare and clustered in summer, when conditions are generally more predictable,
and the temperature component is apparently well enough approximated by the
linear fit across the range Sydney actually reaches.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

In [ ]:
# Translating error into the decision it affects: how often does the
# forecast land in the correct interpretation band?
bands = [0, 20, 40, 60, 80, 100]
band_labels = ["Very poor", "Poor", "Moderate", "Comfortable", "Excellent"]

for h in CCI_HORIZONS:
    dh = datasets[h]
    pred = models[h].predict(dh["X_test"])
    actual_band = pd.cut(dh["y_test"], bins=bands, labels=band_labels, right=False)
    predicted_band = pd.cut(pd.Series(pred, index=dh["y_test"].index),
                            bins=bands, labels=band_labels, right=False)
    agreement = (actual_band == predicted_band).mean()
    within_5 = (np.abs(dh["y_test"] - pred) <= 5).mean()
    print(f"horizon {h}:  correct comfort band {agreement:.1%}   "
          f"within 5 CCI points {within_5:.1%}")

In [ ]:
business_impacts_explanations = """
Translated into the decision it supports, the model places the forecast in
the correct comfort band on 63.3% of one-day-ahead test days, 61.8% at two
days and 61.9% at three. It lands within five CCI points of the truth on
48.9%, 45.5% and 43.2% of days respectively.

Band agreement is the more meaningful figure for an operator deciding
whether to run an outdoor event, because that decision is made on the band
rather than the exact number. Roughly two days in three are categorised
correctly, which supports advisory use but is not strong enough to
schedule against without human judgement. Notably, band agreement barely
degrades between one and three days even though numeric error grows
appreciably, because most of the additional error is absorbed within a
twenty-point band rather than crossing a boundary.

The errors that change decisions are those near a band boundary, where a
small numerical miss flips the recommendation. This argues for the deployed
service exposing the numeric prediction rather than only the band, so that a
value near a boundary is visibly near one.

The most consequential finding for the business is the behaviour on wet
days. The model over-predicts comfort on 98% of days receiving more than
10 mm of rain, by an average of nearly 16 CCI points — frequently enough to
move a day two full bands. This is precisely the failure identified in
section A as the expensive one: customers are brought outdoors into
conditions that were forecast as pleasant, refunds follow and trust erodes.
That it affects only 8% of test days is little comfort, because those are
the days on which a wrong answer costs the most.

The underlying cause is a mismatch between the metric optimised and the cost
borne. Squared error treats over- and under-prediction as equally bad, when
they are not. Three recommendations follow, in order of expected value.
First, and most urgently, the service should not be relied upon for outdoor
commitments when any rainfall is forecast, and the deployed API should be
documented accordingly. Second, a tree-based model should be evaluated, since
it can represent the clipped rain component directly and is likely to
recover much of this error. Third, a quantile or asymmetric loss variant
would let the service offer a deliberately conservative estimate where being
wrong in the optimistic direction is costly.

The model is suitable for advisory use with those caveats stated. It is not
suitable as a sole input to safety-critical decisions, for which the hazard
classification service is the appropriate tool.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

### J.6 Save Model Artefacts

In [ ]:
# Each artefact bundles everything the API needs to reproduce a prediction:
# the fitted pipeline (scaler and model together), the recipe describing how
# to build the features, the exact feature order, and the measured
# performance for /model-metadata.
CCI_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for h in CCI_HORIZONS:
    test_metrics = modeling.evaluate_regression(
        datasets[h]["y_test"], models[h].predict(datasets[h]["X_test"])
    )
    artefact = {
        "pipeline": models[h],
        "recipe": CCI_RECIPE,
        "feature_columns": feature_columns,
        "history_days": features.required_history_days(CCI_RECIPE),
        "target": "Climate Comfort Index",
        "horizon_days": h,
        "algorithm": "Ridge Regression",
        "alpha": best_alphas[h],
        "training_window": [CCI_TRAIN_START, VALIDATION_START],
        "metrics": test_metrics,
        "feature_importance": (
            importance.linear_coefficient_importance(
                models[h].named_steps["model"], feature_columns
            ).head(10).set_index("feature")["coefficient"].round(4).to_dict()
        ),
    }
    path = CCI_MODELS_DIR / f"cci_ridge_h{h}.joblib"
    joblib.dump(artefact, path)
    print(f"saved {path.name}  (test MAE {test_metrics['mae']:.2f})")

## H. Project Outcomes

In [ ]:
# Either 'Hypothesis Confirmed', 'Hypothesis Partially Confirmed' or
# 'Hypothesis Rejected'.
#
# Partially confirmed: the model beats both baselines at every horizon, but
# fails badly on wet days in the specific way a linear model was expected to.
experiment_outcome = "Hypothesis Partially Confirmed"

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
experiment_results_explanations = """
The hypothesis was that a regularised linear model, given lagged weather,
rolling summaries and cyclical calendar features, could forecast the Climate
Comfort Index usefully at one to three days ahead. This is partially
confirmed. The model beats both the persistence and seasonal baselines at
every horizon, reducing MAE from 7.56 to 6.49 at one day and from 9.59 to
7.50 at three, so the feature set carries genuine predictive information
beyond either naive answer. It also fails in a specific and predictable
place: days with meaningful rainfall, where error nearly triples and the
model is optimistic 98% of the time.

Four insights came out of the experiment. First, the two naive baselines are
strong in complementary regimes and weak outside them — persistence at one
day, climatology at three — and the model's value lies in blending them
rather than in beating either outright. Second, that blending is visible in
the fitted coefficients, which shift from today's CCI at one day to the
thirty-day rolling mean and the calendar features at three, reproducing the
persistence-to-climatology transition known from meteorology without it
being imposed. Third, forecast skill decays measurably with horizon, and
this should be surfaced to users rather than hidden behind three
equally-presented numbers. Fourth, and most important for the business, the
average error badly understates the cost of the errors that occur: overall
bias is essentially zero only because a small optimism on dry days cancels a
large one on wet days.

One prediction was wrong and is worth recording. Hot days were expected to
be a second failure mode because the temperature component is non-linear,
but they were forecast better than average. The non-linearity is apparently
mild across the range Sydney actually reaches, which suggests the gain from
moving to a non-linear model will come almost entirely from the rain
component rather than the temperature one.

This is not a dead end and the next steps are clear, ranked by expected
uplift. First, a gradient boosted tree ensemble, which can represent the
clipped rain component directly and is the most likely route to removing the
wet-day error that dominates the loss. Second, rainfall-specific feature
engineering — a binary wet-day indicator, rolling counts of rain days, and
antecedent rainfall over longer windows — since millimetre totals whose
distribution is over half zeros are a poor input regardless of model. Third,
an asymmetric or quantile loss to align the objective with the business cost
of over-predicting comfort. The first two are expected to produce the
largest gains and should be run together as experiment two.

For deployment, the artefacts saved in section J.6 are sufficient to serve
the endpoint: each bundles the fitted model, its scaler, the feature order
and the measured test performance. The recommendation is to deploy this
baseline so the end-to-end service can be validated, then replace the model
artefacts once experiment two has been evaluated, since the API contract
does not change.
"""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)